### Set up the remote kernel
If you use a remote kernel, then you need to set it up with all local code and
the corresponding required dependencies so that we can run our LLM in the Colab
while using the GPU.

In [ ]:
import sys
sys.path

#### Clone repositories

In [ ]:
!git clone https://github.com/NiccoloSacchi/assignment1-basics.git
print()
!git clone https://github.com/NiccoloSacchi/assignment2-systems.git
print()

In [ ]:
# Install Assignment 2's dependencies:
# --system: directly into the environment the Colab kernel is already using.
# -e: in "editable" mode, if you change code in the Assignment 1 folder, the
#   changes are reflected immediately without a reinstall.
!cd assignment2-systems; uv pip install --system -e .

!!!! **Finally, restart the kernel!** !!!!

#### Update repositories

In [ ]:
!cd assignment1-basics; git pull
print()
!cd assignment2-systems; git pull

### GPU

#### Summaries

In [ ]:
import torch
torch.cuda.get_device_properties(0)

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.cuda.memory_summary(device=None, abbreviated=False))

#### Memory release

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

print_gpu_memory()

# Create a tensor on the GPU.
c = 2048
r = 1024
tensor = torch.randn(r, c, dtype=torch.float32, device="cuda")
print(f"Initialized tensor of size {r*c*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete the tensor.
del tensor
print("Deleted tensor.")

# Allocated memory is released but reserved memory is not, it will be used for
# future allocations.
print_gpu_memory()

# Force PyTorch to release all unused reserved memory back to the GPU driver.
torch.cuda.empty_cache()

# Also reserved memory is released.
print_gpu_memory()

#### Arithmetics and backpropagation

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = torch.randn(r, c, dtype=torch.float32, requires_grad=True, device="cuda")
y = torch.randn(c, r, dtype=torch.float32, requires_grad=True, device="cuda")
print(f"Initialized 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB.")
print_gpu_memory()

# Multiply them. There is an additional ~8MB. PyTorch initializes cuBLAS. This
# library loads kernels and allocates internal workspace buffers to handle the
# matrix multiplication efficiently.
z = x @ y
print(f"Multiplied the 2 tensors into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Sum itseld them.
k = z + z
print(f"Summed the tensor into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Backpropagate everything, thus populating grads.
k.sum().backward()
print("Backward propagation.")
print_gpu_memory()

# Delete all tensors and release memory.
del x, y, z, k
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

#### Intermediary results
Intermediary results are stored when grad is enabled. Here we have only 1
variable of  size 4MB, but we consume 20MB.

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = (
  torch.randn(r, c, dtype=torch.float32, requires_grad=True, device="cuda") @
  torch.randn(c, r, dtype=torch.float32, requires_grad=True, device="cuda")
)
print(f"Multiplied in place 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete all tensors and release memory.
del x
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

Intermediary results are *not* stored when requires_grad=False. Here we consume
exactly the size of the output tensor, 4MB.

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = (
  torch.randn(r, c, dtype=torch.float32, requires_grad=False, device="cuda") @
  torch.randn(c, r, dtype=torch.float32, requires_grad=False, device="cuda")
)
print(f"Multiplied in place 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete all tensors and release memory.
del x
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

#### Hard clean

In [ ]:
# Copy-paste the output to Gemini and let it figure out what variable still need
# to be deleted.
locals()

In [ ]:
# Alteratively: restart the kernel.
import os

# Kills the current process and any other python processes on the VM to make
# sure no process is using the GPU anymore.
os.system("pkill -9 python")

### Import and run model

In [ ]:
# Forward pass with an untrained model.
from cs336_basics.model import TransformerLM
import torch

# device=torch.device("cuda")
device = torch.device("mps")
model = TransformerLM(
  vocab_size=10000,
  context_length=256,
  num_layers=8,
  d_model=768,
  num_heads=16,
  d_ff=1344,
  rope_theta=10000.0,
  device=device,
  dtype=torch.float32,
)

input = torch.randint(0, 10000, (1, 256), device=device)
output = model(input)
print(output.shape)

In [ ]:
!cd assignment2-systems; uv run pytest

In [ ]:
# Generate text using a trained model.
from cs336_basics.generation import generate_text
from cs336_basics.tokenizer import BPETokenizer
from cs336_basics.model import TransformerLM
from cs336_basics.io import ROOT_PATH, load_checkpoint

tok = BPETokenizer.load(ROOT_PATH / "tokenizer/tinystories_train_10000.pt")

model, _, _ = load_checkpoint(
  ROOT_PATH / "model/TinyStories/volcanic-dream-4/checkpoint_39999.pt",
  model_class=TransformerLM,
)
text = generate_text(
  modelb=model,
  tokenizer=tok,
  input_text= "Costanza",
  max_new_tokens=100,
  temperature=1.0,
  top_p=0.0,
)
print(model._init_args)
text

### Basic Benchmark

In [ ]:
# Models that we will benchmark.
from dataclasses import dataclass

@dataclass
class ModelConfig:
  vocab_size: int
  d_model: int
  d_ff: int
  num_layers: int
  num_heads: int

BATCH_SIZE = 4
MODELS = {
  "small":  ModelConfig(10000, 768, 3072, 12, 12),
  "medium": ModelConfig(10000, 1024, 4096, 24, 16),
  "large":  ModelConfig(10000, 1280, 5120, 36, 20),
  "xl":     ModelConfig(10000, 1600, 6400, 48, 25),
  "2.7B":   ModelConfig(10000, 2560, 10240, 32, 32),
}

In [ ]:
from cs336_basics.model import TransformerLM
from timeit import default_timer as timer
import numpy as np
import torch

def benchmarking_script(
  model_config: ModelConfig,
  context_length: int,
  measure_also_backward: bool,
  synchronize: bool = True,
  warmup_steps: int = 5,
  measure_steps: int = 10,
  batch_size: int = 4,
  device: torch.device = torch.device("cuda"),
) -> (int, int):
  print("Creating dummy data and model...")
  input = torch.randint(0, model_config.vocab_size, (batch_size, context_length), device=device)
  model = TransformerLM(
    vocab_size=model_config.vocab_size,
    context_length=context_length,
    num_layers=model_config.num_layers,
    d_model=model_config.d_model,
    num_heads=model_config.num_heads,
    d_ff=model_config.d_ff,
    rope_theta=10000.0,
    device=device,
    dtype=torch.float32,
  )
  
  print("Warming up...")
  for _ in range(warmup_steps):
    output = model(input)
    if measure_also_backward:
      output.sum().backward()

  print("Measuring...")
  measures_s = []
  for _ in range(measure_steps):
    # Clear gradients manually to measure 'clean' write speed (avoid readiing
    # and summing gradients from the previous step).
    for p in model.parameters():
      p.grad = None
        
    # Make sure all operations have finished before starting to measure.
    torch.cuda.synchronize()

    start = timer()
    output = model(input)
    if measure_also_backward:
      output.sum().backward()
    if synchronize:
      torch.cuda.synchronize()
    end = timer()

    measures_s.append((end - start))
  return np.mean(measures_s), np.std(measures_s)

In [ ]:
benchmarking_script(
  model_config = MODELS["medium"],
  context_length = 256,
  measure_also_backward = False,
  synchronize = False,
)

In [ ]:
benchmarking_script(
  model_config = MODELS["medium"],
  context_length = 256,
  measure_also_backward = False,
  synchronize = True,
)

In [ ]:
benchmarking_script(
  model_config = MODELS["medium"],
  context_length = 256,
  measure_also_backward = True,
  synchronize = True,
)